# COCO noun–patch routing B — spatial-path control
**Internet ON · T4×2 · Save & Run All.** Một notebook dùng cả hai GPU cho một nhánh.
B/R có thể chạy song song trên hai tài khoản; mỗi tài khoản cần quyền đọc private datasets.
Giữ baseline Q-Former + legacy prompt, không thêm module/parameter inference.
Cùng train10k/dev1000, seed42,1250updates,batch8, CE200 rồi CE+.05SCST(CIDEr-D+.5object_count_v2).
Thêm .05 L_aux từ step1: B dùng khối lượng đường truyền tới mọi patch; R tới patch trong bbox.
Không có kết quả mới trước khi chạy. Nhãn noun–box là giám sát yếu, không phải grounding gold.

In [ ]:
print('COCO ROUTING B: START',flush=True)
from pathlib import Path
ARM='B'
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
NOUN_MANIFEST=Path('/kaggle/input/datasets/ducanh2403/coco-noun-box-train10k-v11')
OUTPUT=Path('/kaggle/working/coco_routing_B')
for path in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,DETECTIONS,NOUN_MANIFEST/'manifest_receipt.json'):
    assert path.is_file(),f'Missing exact input: {path}; sửa path nếu dataset được mount ở tài khoản bạn.'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),'Preserve old output; use fresh session/output.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-coco-routing-b-'))/'repo'
LOG = Path('/tmp/upt_hidden_coco-routing-b.log')
PROJECT_COMMIT = '2bf1e041744f00667fde287baacaeac745462835'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'route_probe.py': '7cd9071bc4671873f027c66e4e243e9771947dba8b4ebe83dae4cc7d129783bc', 'run_route_smoke.py': '1e1d4dd285340d2e8fac5aa069b76c78a079cd8848b8807fcb7a2c3fdd504763', 'routing_data.py': 'd21880bf7b70aba161fd31ef4e10df9fe4ae357a448b995c0f751bee18e0f827', 'routing_model.py': '6388c6a4de9ab0844c6817c803798daaa1c059a3039fd483d2afe8eb5732b735', 'routing_stage.py': '9eab2fa6202298b40b68da2d2c7d6503d847328babde2c1ee67e0d3d73f27a09', 'routing_preflight.py': '9914f95e02e994c2ebdf7a31f3314641c4185bcf2dd64cb8e3f0c6277f5492fa', 'run_routing.py': '0f5a62c5f602c2556e688ea2a6c4a554b5c02611fb7c1da5e4003352144e8c90', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7', 'pilot_model.py': '879046e943169c30e640fd9b04d6a1edfccd19cd17b5311b104b9ec2dc5bf8f1', 'run_pilot.py': 'c34b7d7f2aae7e050190f713370d0c5f85b283bf342e328e63baeaf2783061a1', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'run_hidden.py': 'f12b2202e57bcbf4a237ded68844b4d5e551ef231e8906e1e3096abcfe0e3033', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/upt_reference.py': '22d8172fd5da419a23a6c4f65189e1374c29be2e2f24631f735221896c0a9dac'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
    'h5py','numpy','Pillow','tqdm','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
    'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
CHAIR=Path('/tmp/coco-routing-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
import torch
assert torch.cuda.device_count()==2,'Select GPU T4x2 for this one arm.'
run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)


In [ ]:
CONFIG=Path('/tmp/coco_routing_'+ARM+'.json')
cfg=dict(arm=ARM,checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),detections=str(DETECTIONS),noun_manifest=str(NOUN_MANIFEST),output=str(OUTPUT),
    stage='/tmp/coco_routing_'+ARM+'.h5',chair=str(CHAIR),project_commit=PROJECT_COMMIT,
    implementation_sha256=SOURCE_HASHES,noun_receipt_sha256='bde873ba77124333488b263a052419dc80e412c1e089178cdf423d2836e1b146',models_source_sha256='18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8',
    global_batch_size=8,eval_batch_size=16,warmup_updates=200,base_lr=5e-6,scst_weight=.05,seed=42,
    aux_weight=.05,route_epsilon=1e-8,updates=1250,
    adamw_options=dict(weight_decay=.01,betas=[.9,.999],eps=1e-8,amsgrad=False,maximize=False,
        foreach=None,capturable=False,differentiable=False,fused=None),
    record_optimizer_manifest=True,max_gradient_norm=1.,dropout_policy='model.eval() with gradients enabled',
    locked_numeric_gate=dict(R_minus_A_CIDEr_min=.01,R_minus_A_paired95_lower_gt=0.,
        R_CIDEr_gt_original=True,CHAIRs_increase_max=0.,CHAIRi_increase_max=0.,Recall_drop_max=.01,
        position_attribution_requires_R_minus_B_paired95_lower_gt=0.),
    promotion_requires='Matched protocol, numeric gates, then small blind factual review. No test/rerank/sweep.')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_routing.py','prepare','--config',CONFIG],cwd=REPO)
    for mode in ('preflight','worker'):
        run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
            '--nproc_per_node=2','--max_restarts=0','run_routing.py',mode,'--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_routing.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    import zipfile
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        for name,source in SOURCES.items():
            target=OUTPUT/'source'/name;target.parent.mkdir(parents=True,exist_ok=True)
            target.write_text(source,encoding='utf-8')
        if PROTOCOL.is_file():
            run([sys.executable,'-u','run_routing.py','archive','--config',PROTOCOL],cwd=REPO)
        else:
            with zipfile.ZipFile(str(OUTPUT)+'_review.zip','w',zipfile.ZIP_DEFLATED) as z:
                for path in sorted(OUTPUT.rglob('*')):
                    if path.is_file() and path.suffix in ('.json','.jsonl','.log','.py'):
                        z.write(path,path.relative_to(OUTPUT))
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)


Giữ checkpoint.bin riêng và gửi ZIP review. Stage HDF5 trong /tmp không phải output cần lưu.
Không cần UPT/VG cache, ảnh gốc hay full COCO annotation ZIP. Cần dataset 7 JSON noun-box đã khóa.
Preflight nằm trong notebook, không cập nhật trọng số. FAIL thì gửi log; không bỏ qua/đổi tolerance.
Nếu không thấy lỗi gốc: `print(LOG.read_text(encoding='utf-8')[-16000:])`.
Chờ đủ B/R mới so sánh với A đã có. Dev1000 đã dùng trước; chưa phải bằng chứng test độc lập.